# Microsoft-Authentifizierung

Die Microsoft-Authentifizierung ermöglicht die Anmeldung an Backstage über Microsoft Entra ID beziehungsweise Microsoft 365.

Das Auth-Modul authentifiziert Benutzer und ordnet die externe Microsoft-Identität einer Backstage `User` Entity zu. Kubernetes wird nicht verwendet.


## Microsoft Auth Provider installieren

Das Microsoft Auth Provider Modul wird im Backstage-Backend installiert.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn --cwd packages/backend add @backstage/plugin-auth-backend-module-microsoft-provider


## Backend-Modul registrieren

Das Microsoft Auth Provider Modul wird im Backstage-Backend registriert, damit die Microsoft OAuth Endpunkte beim Start bereitgestellt werden.

Dazu patchen wir die Datei [packages/backend/src/index.ts](../../mybackstage/packages/backend/src/index.ts)


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

grep -q \
  "plugin-auth-backend-module-microsoft-provider" packages/backend/src/index.ts \
|| sed -i "/backend.start();/i backend.add(import('@backstage/plugin-auth-backend-module-microsoft-provider'));" packages/backend/src/index.ts

# Kontrolle
yarn why @backstage/plugin-auth-backend-module-microsoft-provider


## Anwendung in Microsoft Entra ID registrieren

Im **Microsoft Entra Admin Center**:

* **Identity → Applications → App registrations** öffnen
* **New registration** auswählen
* Als Namen beispielsweise `Backstage Microsoft Auth` eintragen
* Den gewünschten Kontotyp auswählen
* Unter **Redirect URI** den Typ **Web** auswählen
* Die unten ausgegebene Redirect URI eintragen
* Anwendung registrieren
* `Application (client) ID` kopieren
* `Directory (tenant) ID` kopieren
* Unter **Certificates & secrets** ein neues Client Secret erzeugen
* Den Secret-**Wert** sofort kopieren
* Werte in [env-platen.py](../../data/env-platen.py) eintragen:
  * `AUTH_MICROSOFT_CLIENT_ID`
  * `AUTH_MICROSOFT_CLIENT_SECRET`
  * `AUTH_MICROSOFT_TENANT_ID`

Die Redirect URI muss exakt auf den Backstage Auth Handler zeigen und darf nach `frame` keinen abschliessenden Slash enthalten.


In [ ]:
%%bash
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)

echo "Microsoft Redirect URI:"
echo "http://localhost:7007/api/auth/microsoft/handler/frame"


## Umgebungsvariablen prüfen

Die OAuth-Zugangsdaten werden aus `env-platen.py` geladen. Das Script zeigt nur, ob die Variablen gesetzt sind, und gibt keine Secrets aus.


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
set +a

for variable in \
  AUTH_MICROSOFT_CLIENT_ID \
  AUTH_MICROSOFT_CLIENT_SECRET \
  AUTH_MICROSOFT_TENANT_ID
do
  if test -n "${!variable}"; then
    echo "${variable} ist gesetzt"
  else
    echo "${variable} fehlt"
  fi
done


## Microsoft Auth Provider konfigurieren

Der Microsoft Provider wird in einer separaten Backstage-Konfigurationsdatei eingerichtet.

Der Resolver `emailMatchingUserEntityProfileEmail` vergleicht die E-Mail-Adresse des Microsoft-Kontos mit `spec.profile.email` einer Backstage `User` Entity.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

cat > app-config.microsoft-auth.yaml <<'EOF'
auth:
  environment: development
  providers:
    microsoft:
      development:
        clientId: ${AUTH_MICROSOFT_CLIENT_ID}
        clientSecret: ${AUTH_MICROSOFT_CLIENT_SECRET}
        tenantId: ${AUTH_MICROSOFT_TENANT_ID}
        signIn:
          resolvers:
            - resolver: emailMatchingUserEntityProfileEmail
EOF


## Microsoft Login im Frontend konfigurieren

Die neue Backstage Frontend-Architektur verwendet eine `SignInPageBlueprint` Extension.

Das folgende Script erstellt die Extension in `packages/app/src/extensions/microsoftSignInPage.tsx`.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

mkdir -p packages/app/src/extensions

cat > packages/app/src/extensions/microsoftSignInPage.tsx <<'EOF'
import { SignInPage } from '@backstage/core-components';
import { microsoftAuthApiRef } from '@backstage/core-plugin-api';
import { SignInPageBlueprint } from '@backstage/plugin-app-react';

export const microsoftSignInPage = SignInPageBlueprint.make({
  params: {
    loader: async () => props => (
      <SignInPage
        {...props}
        provider={{
          id: 'microsoft-auth-provider',
          title: 'Microsoft',
          message: 'Mit Microsoft anmelden',
          apiRef: microsoftAuthApiRef,
        }}
      />
    ),
  },
});
EOF

echo "Extension erstellt:"
echo "packages/app/src/extensions/microsoftSignInPage.tsx"


## Frontend Extension registrieren

Das Script ergänzt den Import und fügt `microsoftSignInPage` am Anfang der vorhandenen `features`-Liste ein.

Vor der Änderung wird eine Sicherung von `App.tsx` erstellt. Falls die erwartete `features`-Liste nicht gefunden wird, bricht das Script ohne Änderung ab.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

python3 - <<'PY'
from pathlib import Path
import shutil

path = Path("packages/app/src/App.tsx")
backup = path.with_suffix(".tsx.microsoft-auth.bak")
text = path.read_text(encoding="utf-8")

import_line = (
    "import { microsoftSignInPage } "
    "from './extensions/microsoftSignInPage';"
)

if import_line not in text:
    lines = text.splitlines()
    last_import = max(
        (index for index, line in enumerate(lines) if line.startswith("import ")),
        default=-1,
    )
    lines.insert(last_import + 1, import_line)
    text = "\n".join(lines) + "\n"

if "microsoftSignInPage," not in text:
    marker = "features: ["
    if marker not in text:
        raise SystemExit(
            "Keine features-Liste gefunden. App.tsx wurde nicht verändert."
        )
    text = text.replace(
        marker,
        marker + "\n    microsoftSignInPage,",
        1,
    )

if not backup.exists():
    shutil.copy2(path, backup)

path.write_text(text, encoding="utf-8")
print(f"App.tsx aktualisiert. Sicherung: {backup}")
PY

grep -n "microsoftSignInPage" packages/app/src/App.tsx


## Backstage User Entity prüfen

Der gewählte Resolver benötigt eine `User` Entity, deren `spec.profile.email` exakt der Microsoft-E-Mail-Adresse entspricht.

Die bestehende User-Konfiguration kann mit dem folgenden Befehl gesucht werden.


In [ ]:
%%bash
cd ~/mybackstage/

grep -R --line-number --include='*.yaml' --include='*.yml' \
  -E '^kind:[[:space:]]*User|^[[:space:]]*email:' \
  examples catalog 2>/dev/null || true


## Konfiguration prüfen

Mit `yarn backstage-cli config:print` wird die zusammengeführte und aufgelöste Backstage-Konfiguration ausgegeben, ohne die Anwendung zu starten.


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Microsoft Auth"
export BACKSTAGE_PORT="3001"

source ~/.nvm/nvm.sh
cd ~/mybackstage

yarn backstage-cli config:print \
  --config ~/mybackstage/app-config.yaml \
  --config ~/mybackstage/app-config.test.yaml \
  --config ~/mybackstage/app-config.microsoft-auth.yaml


## Backstage starten

Der Start-Endpunkt des Microsoft Providers muss mit einem HTTP Redirect antworten.

In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Microsoft Auth"
export BACKSTAGE_PORT="3001"

echo "Frontend: http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
echo "Backend:  http://localhost:7007/api/auth/microsoft/start?env=development"

source ~/.nvm/nvm.sh
cd ~/mybackstage

yarn start \
  --config ~/mybackstage/app-config.yaml \
  --config ~/mybackstage/app-config.test.yaml \
  --config ~/mybackstage/app-config.microsoft-auth.yaml \
  2>&1 | tee /tmp/backstage-microsoft-auth.log


**Links**

- [Microsoft Authentication Provider](https://backstage.io/docs/auth/microsoft/provider/)
- [Authentication in Backstage](https://backstage.io/docs/auth/)
- [Sign-in Identities and Resolvers](https://backstage.io/docs/auth/identity-resolver/)
